<a href="https://colab.research.google.com/github/Apoorvmittal11/23-CS-072-DL-LAB-EXPERIMENT/blob/main/DL%20EXPERIMENT%205/1D_CNN_for_text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install tensorflow

#**Data Loading and Setup**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

print("Loading IMDB Dataset...")
VOCAB_SIZE = 10000
MAX_LEN = 200

# Load pre-tokenized data from Keras
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

# Pad sequences so they are all the same length
X_train = pad_sequences(X_train, maxlen=MAX_LEN, padding='post', truncating='post')
X_test = pad_sequences(X_test, maxlen=MAX_LEN, padding='post', truncating='post')

# Convert to PyTorch Tensors
X_train_t = torch.tensor(X_train, dtype=torch.long)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

# Create DataLoaders
BATCH_SIZE = 64
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=BATCH_SIZE)


Loading IMDB Dataset...


#**Defining the 1D CNN Model**

In [ ]:
class CNN1D_Text(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_filters, kernel_size):
        super(CNN1D_Text, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.conv1d = nn.Conv1d(in_channels=embed_dim,
                                out_channels=num_filters,
                                kernel_size=kernel_size)

        self.relu = nn.ReLU()
        self.fc = nn.Linear(num_filters, 1)

    def forward(self, x):
        embedded = self.embedding(x)

        embedded = embedded.permute(0, 2, 1)
        #Convolution + ReLU
        conved = self.relu(self.conv1d(embedded))

        pooled = torch.max(conved, dim=2)[0]
        out = self.fc(pooled)

        return out.squeeze(1)

#**Initialization & Training Loop**

In [ ]:
EMBED_DIM = 100
NUM_FILTERS = 128
KERNEL_SIZE = 5
EPOCHS = 5

model = CNN1D_Text(VOCAB_SIZE, EMBED_DIM, NUM_FILTERS, KERNEL_SIZE).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("\n--- Starting Training ---")
for epoch in range(EPOCHS):
    model.train()
    total_loss, total_correct = 0, 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        predictions = model(batch_X)

        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        rounded_preds = torch.round(torch.sigmoid(predictions))
        total_correct += (rounded_preds == batch_y).sum().item()

    avg_loss = total_loss / len(train_loader)
    avg_acc = total_correct / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_loss:.4f} | Train Acc: {avg_acc*100:.2f}%")


--- Starting Training ---
Epoch 1/5 | Train Loss: 0.5268 | Train Acc: 72.96%
Epoch 2/5 | Train Loss: 0.3415 | Train Acc: 85.68%
Epoch 3/5 | Train Loss: 0.2202 | Train Acc: 92.30%
Epoch 4/5 | Train Loss: 0.1253 | Train Acc: 96.80%
Epoch 5/5 | Train Loss: 0.0610 | Train Acc: 99.33%


#**Performance Evaluation**

In [ ]:

model.eval()
test_loss, test_correct = 0, 0

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        predictions = model(batch_X)

        loss = criterion(predictions, batch_y)
        test_loss += loss.item()

        rounded_preds = torch.round(torch.sigmoid(predictions))
        test_correct += (rounded_preds == batch_y).sum().item()

final_test_loss = test_loss / len(test_loader)
final_test_acc = test_correct / len(test_loader.dataset)

print("\n--- Final Results ---")
print(f"Test Loss: {final_test_loss:.4f} | Test Accuracy: {final_test_acc*100:.2f}%")


--- Final Results ---
Test Loss: 0.3760 | Test Accuracy: 85.12%
